# Explicações pré-código

A parte do código da Sprint 10 foi mantida em inglês porque já fazia parte do meu repositório (que é todo em inglês porque busco transição de carreira como Data Scientist no UUEE)

Segue link do repositório deste projeto:
https://github.com/pedrolocosta/bank-customer-churn-prediction

Nesta versão estou te mandando meu requirements.txt na célula abaixo porque irei utilizar o imblearn que não roda aqui na plataforma TripleTen.
A decisão de usar este recurso foi porque o balanceamento feito com o pandas como fiz na versão Sprint 10 leva em conta apenas os múltiplos inteiros de repetição. Então ao invés de buscar uma alternativa para múltiplos fracionados, eu preferia usar o imblearn que é que eu faria na prática.

# Bank Customer Churn Prediction — TripleTen Sprint 10 with update on Sprint 12
# Install in a Python 3.12 Conda environment:
#   python -m pip install -r requirements.txt
#
# These are compatibility ranges, NOT an exact environment lockfile.
# Record the versions from a successful notebook run if exact reproducibility is needed.

# Data analysis
numpy>=1.26,<3
pandas>=2.1,<3

# Machine learning and evaluation
scikit-learn>=1.4,<2

# Data visualization
matplotlib>=3.8,<4
seaborn>=0.13,<1

# VS Code / Jupyter kernel
ipykernel>=6.29,<8


# Inicialization

In [61]:
# Import project libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
#from sklearn.utils import shuffle
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

In [62]:
# Configure display settings for this analysis
pd.set_option('display.max_colwidth', None)

## Loading the dataset

In [63]:
# Loading the dataset
df = pd.read_csv('datasets/Churn.csv')

## Adding custom functions

In [64]:
def std_col_names(original_df:pd.DataFrame):

    '''Apply a standard cleaning procedure to the original DataFrame's column 
    names by converting them to lowercase, removing extra whitespace, and 
    replacing spaces with underscores to follow the `snake_case` naming 
    convention. Then, replace the original column names with the cleaned ones.'''

    old_col_names = original_df.columns.tolist()
    
    new_col_names = []
    for old_name in old_col_names:
        name_stripped = old_name.strip()
        name_lowered = name_stripped.lower()
        name_no_space = name_lowered.replace(' ', '_')
        new_col_names.append(name_no_space)

    original_df.columns = new_col_names

    return print('Lista dos novos nomes das colunas:\n', original_df.columns)

In [65]:
def assess_type(dataframe:pd.DataFrame, num_rows:int):

    '''Creates a DataFrame with the number of rows determinated by the 
    'num_row' variable, and sets the last row show the data type of the
     values in that column.'''

    assess_df = pd.concat([
        dataframe.sample(num_rows),
        dataframe.dtypes.to_frame().T.rename(index={0: 'Data Type'})
    ])

    return display(assess_df)

# Limpeza e pré-processamento dos dados

## Initial Assessment

In [66]:
# Inspect dataset structure
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), str(3)
memory usage: 1.1 MB


In [67]:
# Inspect a sample of records
assess_type(df, 5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
7129,7130,15781875,Jamieson,850,Spain,Male,33,3.0,100476.46,2,1,1,136539.13,0
1082,1083,15663438,Andrejew,688,Spain,Male,36,0.0,89772.3,1,1,0,177383.68,1
9206,9207,15611767,Mai,624,Germany,Female,52,0.0,133723.43,1,0,0,4859.59,1
6572,6573,15574237,Hsueh,588,France,Female,21,8.0,0.0,2,1,1,110114.19,0
469,470,15749167,Fisk,753,France,Male,35,3.0,0.0,2,1,1,184843.77,0
Data Type,int64,int64,str,int64,str,str,int64,float64,float64,int64,int64,int64,float64,int64


In [68]:
# Inspect initial descriptive statistics
df.describe()

,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.00000,1.000000e+04,10000.000000,10000.000000,9091.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,5000.50000,1.569094e+07,650.528800,38.921800,4.997690,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,96.653299,10.487806,2.894723,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,584.000000,32.000000,2.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,10000.00000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


#### Resumo da avaliação inicial do dataset

O conjunto de dados consiste de 10 mil linhas e 14 colunas. Os dados são de fácil entendimento. É necessário avaliar com o cliente se a coluna `RowNumber` tem alguma relevância para ele para este estudo ela será removida para reduzir o tamanho do arquivo. Como um aprimoramento deste estudo, a coluna `Surname` também será removida para reduzir o tamanho do arquivo tendo em vista que também é uma coluna identificadora sem relação com o comportamento do cliente; e neste projeto serão adicionadas técnicas que poderão aumentar a quantidade de colunas. A coluna `CustomerId` será mantida para preservar um identificador único para cada linha.

#### Planejamento da limpeza e pré-processamento dos dados


- Remover as colunas: `RowNumber` e `Surname`
- Ajustar os nomes das colunas: aplicar procedimento de limpeza e ajuste padrão (tudo em minúsculo, limpar espaços em branco que houver) e separar palavras por '_'.
- Avaliar valores ausêntes na coluna 'Tenure'
- Avaliar tipo de dado da coluna 'Tenure'
- Avaliar linhas completamente duplicadas
- Avaliar clientes duplicados na coluna 'CustomerId'

## Cleaning and Preprocessing

#### Dropping the `RowNumber`, `CustomerId` and `Surname` columns

In [69]:
# Remove the RowNumber column
df = df.drop(['RowNumber', 'Surname'], axis=1)

In [70]:
# Confirm columns were removed
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerId       10000 non-null  int64  
 1   CreditScore      10000 non-null  int64  
 2   Geography        10000 non-null  str    
 3   Gender           10000 non-null  str    
 4   Age              10000 non-null  int64  
 5   Tenure           9091 non-null   float64
 6   Balance          10000 non-null  float64
 7   NumOfProducts    10000 non-null  int64  
 8   HasCrCard        10000 non-null  int64  
 9   IsActiveMember   10000 non-null  int64  
 10  EstimatedSalary  10000 non-null  float64
 11  Exited           10000 non-null  int64  
dtypes: float64(3), int64(7), str(2)
memory usage: 937.6 KB
None


#### Standardizing Column Names

In [71]:
# Inspect current column names
print('Lista de nome das colunas:\n', df.columns)

Lista de nome das colunas:
 Index(['CustomerId', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure',
       'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember',
       'EstimatedSalary', 'Exited'],
      dtype='str')


In [72]:
# Standardize column names
std_col_names(df)

Lista dos novos nomes das colunas:
 Index(['customerid', 'creditscore', 'geography', 'gender', 'age', 'tenure',
       'balance', 'numofproducts', 'hascrcard', 'isactivemember',
       'estimatedsalary', 'exited'],
      dtype='str')


In [73]:
# renomeando as colunas com o padrão snake_case
df = df.rename(columns={
    'customerid': 'customer_id',
    'creditscore': 'credit_score',
    'numofproducts': 'num_of_products',
    'hascrcard': 'has_cr_card',
    'isactivemember': 'is_active_member',
    'estimatedsalary': 'estimated_salary'
})

print('Nome das colunas:\n', df.columns)

Nome das colunas:
 Index(['customer_id', 'credit_score', 'geography', 'gender', 'age', 'tenure',
       'balance', 'num_of_products', 'has_cr_card', 'is_active_member',
       'estimated_salary', 'exited'],
      dtype='str')


#### Assessing Missing Values

In [74]:
# Check for missing values
print('Missing values by column:')
print(df.isna().sum())

Missing values by column:
customer_id           0
credit_score          0
geography             0
gender                0
age                   0
tenure              909
balance               0
num_of_products       0
has_cr_card           0
is_active_member      0
estimated_salary      0
exited                0
dtype: int64


##### Examining Missing Values in `tenure`

In [75]:
# Inspect tenure before deciding how to handle missing values
# Inspect the tenure data
df['tenure'].value_counts()

tenure
1.0     952
2.0     950
8.0     933
3.0     928
5.0     927
7.0     925
4.0     885
9.0     882
6.0     881
10.0    446
0.0     382
Name: count, dtype: int64

In [76]:
# Avaliando os dados estatístico da coluna `tenune`
print('Descriptive statistics for tenure:')
df['tenure'].agg(['mean', 'median', 'min', 'max', 'skew'])

Descriptive statistics for tenure:


mean       4.997690
median     5.000000
min        0.000000
max       10.000000
skew       0.016153
Name: tenure, dtype: float64

#### Checking data type

In [77]:
assess_type(df, 4)

,customer_id,credit_score,geography,gender,age,tenure,balance,num_of_products,has_cr_card,is_active_member,estimated_salary,exited
5367,15742848,673,France,Male,41,5.0,0.0,1,1,1,65657.29,0
7753,15777616,605,Germany,Male,28,10.0,113690.83,1,1,0,33114.24,0
9311,15608381,585,Germany,Male,50,2.0,125845.66,1,1,0,9439.31,1
5392,15710012,738,Spain,Male,44,2.0,0.0,2,1,0,43018.82,1
Data Type,int64,int64,str,str,int64,float64,float64,int64,int64,int64,float64,int64


##### Adjusting the data type of the `tenure` column

In [78]:
# Convert tenure to nullable integer
df['tenure'] = df['tenure'].astype('Int64')

In [79]:
# Confirming the data type of the `tenure` column
print('tenure data type:', df['tenure'].dtype)

tenure data type: Int64


#### Checking Duplicated Rows

In [80]:
# Check for fully duplicated rows
print('Number of duplicated rows:', df.duplicated().sum())

Number of duplicated rows: 0


##### Checking for Duplicate Customers in `customer_id`

In [81]:
# Check unique customer IDs
print('Unique customer_id values:',
      df['customer_id'].nunique())
print('Dataset row count:', df.shape[0])

Unique customer_id values: 10000
Dataset row count: 10000


#### Resumo da limpeza e pré-processamento dos dados

**Remover colunas:** As colunas `RowNumber`, e `Surname` foram removidas por não ter relação com o comportamento do cliente, logo não são representativas para este estudo. A coluna `CustomerID` foi mantida para preservar um identificado único para cada linha.

**Ajuste no nome das colunas:** Os títulos das colunas foram ajustados com uma função customizada sempre é aplicada nos projetos. Dessa forma foram abrangidas as situações mais comuns como a retirada de possíveis faixas ou espaços em branco, todos os caracteres foram transformados em letra minúscula, possíveis espaços em branco que houvessem foram substituídos por '_'. Adicionalmente, para este projeto foi feito a renomeação dos títulos que foram necessários para o padrão snake_case.

**Dados ausentes:** Foram encontrados dados ausentes apenas na coluna `tenure`. Para o preenchimento destes dados ausentes, temos que a média (4,99 anos) e a mediana (5,00 anos) observadas são praticamente idênticas. A assimetria é aproximadamente de 0,016, indicando um baixo grau de assimetria. A imputação pela mediana foi escolhida como referência simples, embora a imputação pela média fornecesse um valor de substituição semelhante. Os valores ausentes serão preenchidos após a divisão dos dados. A mediana será calculada utilizando apenas o conjunto de treinamento e aplicada, sem alterações, aos conjuntos de treinamento, validação e teste, a fim de evitar o vazamento de dados.

**Tipo de dados:** Apenas na coluna `tenure` o tipo do dado contido nela precisava ser ajustado. Como a coluna `tenure` avalia o período de maturação em anos, e todos os dados inseridos eram números inteiros, esta coluna teve o seu tipo de dado alterado para 'Int64'.

**Dados duplicados:** não foi encontrada nenhuma linha completamente duplicada. Adicionalmente não foi encontrado nenhum indicador exclusivo de clientes repetido na coluna `customer_id`.

# Defining Features and Target

In [82]:
# Separate features from the target
y = df['exited']
X = df.drop(['exited', 'customer_id'], axis=1)

# Creating Reproducible Data Splits

### First Split: Reserving 40% for Validation and Testing

In [83]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=67, stratify=y
)

### Second Split: Dividing the Reserved 40% Equally

In [84]:
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=67, stratify=y_temp
)

### Checking Split Sizes, Class Proportions, and Missing `tenure` Values

In [85]:
for name, split_X, split_y in [
    ('Training', X_train, y_train),
    ('Validation', X_valid, y_valid),
    ('Test', X_test, y_test)
]:
    print(f'{name}: {len(split_X)} linhas | '
          f'Rotatividade de clientes: {split_y.mean():.2%} | '
          f'Missing values in tenure: {split_X["tenure"].isna().sum()}'
          )

Training: 6000 linhas | Rotatividade de clientes: 20.37% | Missing values in tenure: 544
Validation: 2000 linhas | Rotatividade de clientes: 20.35% | Missing values in tenure: 183
Test: 2000 linhas | Rotatividade de clientes: 20.40% | Missing values in tenure: 182


# Processamento dos dados

### Planejamento do processamento dos dados

- Fase 01 - Tratamento dos valores ausentes
- Fase 02 - Tratamento das variáveis categóricas
- Fase 03 - Interação e Polinomial Features
- Fase 04 - Escalonamento das features
- Fase 05 - Balanceamento de classes (neste projeto será feito após o teste dos modelos sem balancear)

## Tratamento dos valores ausentes

### Tratando os valores ausentes

In [86]:
# Calculate median tenure using the training set
tenure_median = X_train['tenure'].median()

In [ ]:
for name, split_X in [
    ('training', X_train),
    ('validação', X_valid),
    ('teste', X_test)   
]:
    # Impute missing tenure values using the training-set median
    split_X['tenure'] = (
        split_X['tenure'].astype('float64').fillna(tenure_median)
    )

### Checking Split Sizes, Class Proportions, and Missing `tenure` Values

In [90]:
for name, split_X, split_y in [
    ('Training', X_train, y_train),
    ('Validation', X_valid, y_valid),
    ('Test', X_test, y_test)
]:
    print(f'{name}: {len(split_X)} linhas | '
          f'Rotatividade de clientes: {split_y.mean():.2%} | '
          f'Missing values in tenure: {split_X["tenure"].isna().sum()}'
          )

Training: 6000 linhas | Rotatividade de clientes: 20.37% | Missing values in tenure: 0
Validation: 2000 linhas | Rotatividade de clientes: 20.35% | Missing values in tenure: 0
Test: 2000 linhas | Rotatividade de clientes: 20.40% | Missing values in tenure: 0


## Tratamento das variáveis categóricas

### Avaliação das features

In [91]:
# Avaliando o tipo de dado de cada feature
assess_type(df,4)

,customer_id,credit_score,geography,gender,age,tenure,balance,num_of_products,has_cr_card,is_active_member,estimated_salary,exited
7982,15665129,545,Germany,Male,33,1,132527.9,2,0,1,107429.71,0
5163,15633870,850,France,Female,36,10,0.0,2,1,1,100750.03,0
57,15647091,725,Germany,Male,19,0,75888.2,1,0,0,45613.75,0
6360,15661591,413,Germany,Male,39,1,130969.77,2,1,1,158891.79,0
Data Type,int64,int64,str,str,int64,Int64,float64,int64,int64,int64,float64,int64


In [92]:
# Identificando quais variáveis estão sendo consideradas numéricas
df.describe()

,customer_id,credit_score,age,tenure,balance,num_of_products,has_cr_card,is_active_member,estimated_salary,exited
count,1.000000e+04,10000.000000,10000.000000,9091.0,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,1.569094e+07,650.528800,38.921800,4.99769,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,7.193619e+04,96.653299,10.487806,2.894723,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.556570e+07,350.000000,18.000000,0.0,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,1.562853e+07,584.000000,32.000000,2.0,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,1.569074e+07,652.000000,37.000000,5.0,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,1.575323e+07,718.000000,44.000000,7.0,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,1.581569e+07,850.000000,92.000000,10.0,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


#### Resumo da avaliação das features

Apenas as variáveis das colunas `geography` e `gender` são não numéricas da categoria nominal. Não há variáveis da categoria ordinal.

#### Planejamento do processamento das variáveis categóricas

Criar dois novos DataFrames com tratamento diferentes para aplicação em modelos de ML diferentes: 
- No df que será direcionado para modelos não baseados em árvores será aplicado OHE nas colunas `geography` e `gender`.
- No df que será direcionado para modelos baseados em árvores, será aplicado o Label Encoding nas colunas `geography` e `gender`. 

### Ajustar as variáveis categóricas

#### Criar df para algorítimos não baseados em árvore

In [108]:
# codificando as colunas categóricas com one-hot encoding

# dicionário para armazenar os conjuntos de features codificados
encoded_X = {}

for name, split_X in [
    ('training', X_train),
    ('validação', X_valid),
    ('teste', X_test)   
]:
    prepared_X = split_X.copy()
    
    # One-hot encode the gender column
    encoded_gender = pd.get_dummies(
        prepared_X, columns=['gender'], drop_first=True,
        dtype=int
    )

    encoded = encoded_gender.copy()

    # One-hot encode the geography column
    encoded = pd.get_dummies(
        encoded_gender, columns=['geography'], drop_first=True,
        dtype=int
    )

    # Align validation and test columns with training columns
    if name != 'training':
        encoded = encoded.reindex(
            columns=encoded_X['training'].columns,
            fill_value=0
        )

    # Store the encoded feature split in the dictionary
    encoded_X[name] = encoded

In [109]:
# Assign encoded feature splits to separate variables
X_train_ohe = encoded_X['training']
X_valid_ohe = encoded_X['validação']
X_test_ohe = encoded_X['teste']

### Checking and Comparing the Newly Created Columns

In [115]:
for name, split_X in [
    ('Training', X_train),
    ('Treinamento OHE', X_train_ohe),
    ('Validation', X_valid),
    ('Validation OHE', X_valid_ohe),
    ('Test', X_test),
    ('Test OHE', X_test_ohe)
]:
    print(f'{name}: {split_X.shape[1]} columns')
    print(', '.join(split_X.columns), '\n')

Training: 10 columns
credit_score, geography, gender, age, tenure, balance, num_of_products, has_cr_card, is_active_member, estimated_salary 

Treinamento OHE: 11 columns
credit_score, age, tenure, balance, num_of_products, has_cr_card, is_active_member, estimated_salary, gender_Male, geography_Germany, geography_Spain 

Validation: 10 columns
credit_score, geography, gender, age, tenure, balance, num_of_products, has_cr_card, is_active_member, estimated_salary 

Validation OHE: 11 columns
credit_score, age, tenure, balance, num_of_products, has_cr_card, is_active_member, estimated_salary, gender_Male, geography_Germany, geography_Spain 

Test: 10 columns
credit_score, geography, gender, age, tenure, balance, num_of_products, has_cr_card, is_active_member, estimated_salary 

Test OHE: 11 columns
credit_score, age, tenure, balance, num_of_products, has_cr_card, is_active_member, estimated_salary, gender_Male, geography_Germany, geography_Spain 



#### Criar df para algorítimo baseado em árvore

# Training Models on Imbalanced Data

**Plan**

- Examine the target-class distribution.
- Define the features and target.
- Split the data reproducibly into training, validation, and test sets.
- Prepare features and compare baseline classification models.

## Evaluating Class Imbalance

In [ ]:
# Create a DataFrame to assess the exited class distribution
class_balance = pd.DataFrame({
    'count': df['exited'].value_counts(),
    'percentege': df['exited'].value_counts(normalize=True) * 100
}).sort_index()

display(class_balance.round(2))

In [ ]:
# Plot the target-class distribution

fig, ax = plt.subplots(figsize=(6,4))

bars = ax.bar(
    ['Permaneceram (0)', 'Saíram (1)'],
    class_balance['count'],
    color=['steelblue', 'darkorange']
)

ax.set(
    title='Customer churn: class distribution',
    xlabel='Saída de clientes',
    ylabel='Number of customers'
)
ax.set_ylim(0, class_balance['count'].max()*1.15)

plt.tight_layout()
plt.show()

## Preprocessing Data for the Models

### Handling Missing Values and Categorical Features

In [ ]:
# Impute missing tenure values using the training median
# codificando as colunas categóricas com one-hot encoding

# dicionário para armazenar os conjuntos de features codificados
encoded_features = {}

for name, split_features in [
    ('training', features_train),
    ('validação', features_valid),
    ('teste', features_test)   
]:
    prepared_features = split_features.copy()

    # Impute missing tenure values using the training-set median
    prepared_features['tenure'] = (
        prepared_features['tenure'].astype('float64').fillna(tenure_median)
    )

    # One-hot encode the gender column
    encoded_gender = pd.get_dummies(
        prepared_features, columns=['gender'], drop_first=True,
        dtype=int
    )

    encoded = encoded_gender.copy()

    # One-hot encode the geography column
    encoded = pd.get_dummies(
        encoded_gender, columns=['geography'],
        dtype=int
    )

    # Align validation and test columns with training columns
    if name != 'training':
        encoded = encoded.reindex(
            columns=encoded_features['training'].columns,
            fill_value=0
        )

    # Store the encoded feature split in the dictionary
    encoded_features[name] = encoded


In [ ]:
# Assign encoded feature splits to separate variables
features_train_encoded = encoded_features['training']
features_valid_encoded = encoded_features['validação']
features_test_encoded = encoded_features['teste']

### Scaling Numerical Features for Logistic Regression

In [ ]:
# Select numerical columns
numeric_columns = [
    'credit_score',
    'age',
    'tenure',
    'balance',
    'num_of_products',
    'estimated_salary'
]

In [ ]:
# Fit scaling parameters on the training set
scaler = StandardScaler()
scaler.fit(features_train_encoded[numeric_columns])

In [ ]:
# Scale numerical features across encoded splits
scaled_features = {}

for name, split_features in [
    ('training', features_train_encoded),
    ('validação', features_valid_encoded),
    ('teste', features_test_encoded)
]:
    scaled = split_features.copy()

    scaled[numeric_columns] = scaler.transform(
        split_features[numeric_columns]
    )

    scaled_features[name] = scaled

In [ ]:
# Assign scaled feature splits to separate variables
features_train_scaled = scaled_features['training']
features_valid_scaled = scaled_features['validação']
features_test_scaled = scaled_features['teste']

In [ ]:
display(
    features_train_scaled[numeric_columns]
    .describe()
    .round(3)
)

## Training Baseline Models Without Class Balancing

### Training Logistic Regression on Scaled Features

In [ ]:
# Initialize the Logistic Regression baseline
logistic_model = LogisticRegression(
    random_state=67, 
    solver='liblinear',
    max_iter=1000
)

# Fit Logistic Regression on the training set
logistic_model.fit(features_train_scaled, target_train)

### Training Random Forest on Encoded, Unscaled Features

In [ ]:
forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=67,
    class_weight=None
)

forest_model.fit(features_train_encoded, target_train)

### Evaluating Both Models with the Same Metrics

In [ ]:
results = []

for name, model, validation_features in [
    ('Logistic Regression', logistic_model, features_valid_scaled),
    ('Random Forest', forest_model, features_valid_encoded)
]:
    predictions = model.predict(validation_features)
    probabilities = model.predict_proba(validation_features)[:, 1]

    results.append({
        'model': name,
        'accuracy': accuracy_score(target_valid, predictions),
        'precision': precision_score(target_valid, predictions, zero_division=0),
        'recall': recall_score(target_valid, predictions, zero_division=0),
        'f1': f1_score(target_valid, predictions, zero_division=0),
        'roc_auc': roc_auc_score(target_valid, probabilities)
    })

    matrix = pd.DataFrame(
        confusion_matrix(target_valid, predictions),
        index=['Actually Stayed', 'Actually Left'],
        columns=['Predicted Stayed', 'Predicted Left']
    )

    print('Matriz de Confusão do modelo de', name)
    display(matrix)

results_df = pd.DataFrame(results).set_index('model')

print('\nModel evaluation results on the validation set:')
display(results_df.round(3))

## Evaluating the Unbalanced Baseline Models

**Class distribution:** The target is imbalanced: 7,963 customers (79.63%) stayed, while 2,037 (20.37%) left. Accuracy alone can therefore be misleading.

**Feature preparation:** Customer identifiers and surnames were excluded from the predictors. The data was split into training (60%), validation (20%), and test (20%) subsets using stratification. Missing `tenure` values were imputed with the training-set median, categorical variables were one-hot encoded, and numerical features were scaled for Logistic Regression.

**Baseline comparison:** Logistic Regression produced a validation F1-score of **0.289** and ROC-AUC of **0.758**. Random Forest performed better, with an F1-score of **0.581** and ROC-AUC of **0.845**.

**Interpretation:** Neither baseline model exceeded the required F1-score of 0.59. Random Forest was the stronger starting point, but class-imbalance strategies and hyperparameter tuning were needed to improve the balance between precision and recall.

# Improving Model Performance

### Comparing Three Class-Balancing Strategies Against the Baseline

In [ ]:
results_balancing = []
trained_model = {}

for model_name, training_features,validation_features in [
    ('Logistic Regression', features_train_scaled, features_valid_scaled),
    ('Random Forest', features_train_encoded, features_valid_encoded)
]:
    for approach in [
        'Baseline',
        'Oversampling',
        'Undersampling',
        'Class weighting'
    ]:
        # Start each experiment with the original training set
        features_fit = training_features
        target_fit = target_train
        class_weight = None

        if approach == 'Oversampling':
            features_zeros = training_features[target_train == 0]
            features_ones = training_features[target_train == 1]
            target_zeros = target_train[target_train == 0]
            target_ones = target_train[target_train ==1]

            features_fit = pd.concat([features_zeros] + [features_ones] * round(len(features_zeros) / len(features_ones)))
            target_fit = pd.concat([target_zeros] + [target_ones] * round(len(target_zeros) / len(target_ones)))
            
            features_fit, target_fit = shuffle(features_fit, target_fit, random_state=67)
            
        elif approach == 'Undersampling':
            features_zeros = training_features[target_train == 0]
            features_ones = training_features[target_train == 1]
            target_zeros = target_train[target_train == 0]
            target_ones = target_train[target_train == 1]

            features_fit = pd.concat(
                [features_zeros.sample(
                    frac=(len(features_ones) / len(features_zeros)), random_state=67
                )] + [features_ones]
            )
            
            target_fit = pd.concat(
                [target_zeros.sample(
                    frac=(len(target_ones) / len(target_zeros)), random_state=67
                )] + [target_ones]
            )
                        
            features_fit, target_fit = shuffle(
                features_fit, target_fit, random_state=67
            )
                      
        elif approach == 'Class weighting':
            class_weight = 'balanced'

        # Initialize a fresh model for this experiment
        if model_name == 'Logistic Regression':
            model = LogisticRegression(
                solver='liblinear',
                max_iter=1000,
                random_state=67,
                class_weight=class_weight
            )
        else:
            model = RandomForestClassifier(
                n_estimators=100,
                random_state=67,
                class_weight=class_weight
            )

        model.fit(features_fit, target_fit)

        # Evaluate the trained model on validation data
        predictions = model.predict(validation_features)
        probabilities = model.predict_proba(validation_features)[:, 1]

        results_balancing.append({
            'model': model_name,
            'approach': approach,
            'training rows': len(target_fit),
            'accuracy': accuracy_score(target_valid, predictions),
            'precision': precision_score(target_valid, predictions, zero_division=0),
            'recall': recall_score(target_valid, predictions, zero_division=0),
            'f1': f1_score(target_valid, predictions, zero_division=0),
            'roc_auc': roc_auc_score(target_valid, probabilities)
        })
        
        trained_model[f'{model_name} - {approach}'] = model

## Comparing Class-Balancing Results

In [ ]:
balancing_results_df = pd.DataFrame(results_balancing)

display(
    balancing_results_df
    .sort_values('f1', ascending=False)
    .round(3)
)

### Finding the Best Hyperparameter Configuration

In [ ]:
# Prepare hyperparameter candidates
forest_params = []

for est in range(50, 101, 10):
    for depth in [5, 10, None]:
        for leaf in [1,3]:
            forest_params.append({
                'n_estimators': est,
                'max_depth': depth,
                'min_samples_leaf': leaf
            })

logistic_params = [
    {'C': 0.01, 'solver':'liblinear'},
    {'C': 0.1, 'solver':'liblinear'},
    {'C': 1, 'solver':'liblinear'},
    {'C': 10, 'solver':'liblinear'}
]

In [ ]:
# Initialize tracking variables
tuning_results = []

best_f1 = {
    'Logistic Regression': -1,
    'Random Forest': -1
}

best_models = {}
best_settings = {}

In [ ]:
# Extend the experiment with hyperparameter search
for model_name, training_features,validation_features in [
    ('Logistic Regression', features_train_scaled, features_valid_scaled),
    ('Random Forest', features_train_encoded, features_valid_encoded)
]:
    # Select candidate parameters for each model
    if model_name == 'Logistic Regression':
         params_options = logistic_params
    else:
         params_options = forest_params
    
    for approach in [
        'Baseline',
        'Oversampling',
        'Undersampling',
        'Class weighting'
    ]:
        # Start each experiment with the original training set
        features_fit = training_features
        target_fit = target_train
        class_weight = None

        if approach == 'Oversampling':
            features_zeros = training_features[target_train == 0]
            features_ones = training_features[target_train == 1]
            target_zeros = target_train[target_train == 0]
            target_ones = target_train[target_train ==1]
    
            features_fit = pd.concat([features_zeros] + [features_ones] * round(len(features_zeros) / len(features_ones)))
            target_fit = pd.concat([target_zeros] + [target_ones] * round(len(target_zeros) / len(target_ones)))
    
            features_fit, target_fit = shuffle(features_fit, target_fit, random_state=67)
    
        elif approach == 'Undersampling':
            features_zeros = training_features[target_train == 0]
            features_ones = training_features[target_train == 1]
            target_zeros = target_train[target_train == 0]
            target_ones = target_train[target_train == 1]
    
            features_fit = pd.concat(
                [features_zeros.sample(
                    frac=(len(features_ones) / len(features_zeros)), random_state=67
                )] + [features_ones]
            )
            target_fit = pd.concat(
                [target_zeros.sample(
                    frac=(len(target_ones) / len(target_zeros)), random_state=67
                )] + [target_ones]
            )
                        
            features_fit, target_fit = shuffle(
                features_fit, target_fit, random_state=67
            )
        
        elif approach == 'Class weighting':
            class_weight = 'balanced'

        # Initialize a fresh model for this experiment
        for params in params_options:
             if model_name == 'Logistic Regression':
                  model = LogisticRegression(
                       max_iter=1000,
                       class_weight=class_weight,
                       random_state=67,
                       **params
                  )
             else:
                model = RandomForestClassifier(
                     random_state=67,
                     class_weight=class_weight,
                     **params
                )

             model.fit(features_fit, target_fit)

             # Evaluate the trained model on validation data
             predictions = model.predict(validation_features)
             probabilities = model.predict_proba(validation_features)[:, 1]

             score = f1_score(target_valid, predictions, zero_division=0)

             result = {
                'model': model_name,
                'approach': approach,
                'parameters': params.copy(),
                'f1': score,
                'precision': precision_score(target_valid, predictions, zero_division=0),
                'recall': recall_score(target_valid, predictions, zero_division=0),
                'roc_auc': roc_auc_score(target_valid, probabilities)
             }

             tuning_results.append(result)
                        
             if score > best_f1[model_name]:
                best_f1[model_name] = score
                best_models[model_name] = model
                best_settings[model_name] = result.copy()


In [ ]:
# Inspect tuning results
tuning_results_df = pd.DataFrame(tuning_results)

display(
    tuning_results_df
    .sort_values('f1', ascending=False, kind='stable')
    .head(10)
    .round(3)
)

In [ ]:
# Display the best F1 result for each model
for model_name, settings in best_settings.items():
    print(f'\nBest {model_name}')
    print('Approach:', settings['approach'])
    print('Parameters', settings['parameters'])
    print(f'Validation F1: {settings["f1"]:.3f}')
    print(f'Validation ROC-AUC: {settings["roc_auc"]:.3f}')

### Assessing the Improvements

**Class-balancing approaches:** Three strategies were compared with the original training distribution: random oversampling, random undersampling, and class weighting. For Random Forest, balanced class weights improved validation F1-score from **0.581** to **0.634**. Recall increased from **0.479** to **0.627**, while precision decreased from **0.739** to **0.641**.

**Hyperparameter selection:** Logistic Regression did not reach the project's required F1-score above 0.59; its best validation F1-score was **0.497**. Several Random Forest configurations met the requirement. The highest validation F1-score was achieved by **Random Forest with balanced class weights**, using `n_estimators=100`, `max_depth=None`, and `min_samples_leaf=1`.

**Metric interpretation:** F1-score is the harmonic mean of precision and recall. Configurations with similar F1-scores can still have different precision–recall trade-offs, which matter for retention decisions.

**ROC-AUC:** The selected Random Forest achieved a validation ROC-AUC of **0.847**, substantially above the random-classifier reference of 0.5. ROC-AUC complements F1-score by measuring the model's ability to rank customers by churn risk across classification thresholds.

# Final Test Evaluation

In [ ]:
# Initialize the final model with selected parameters
final_model = RandomForestClassifier(
        random_state=67,
        class_weight='balanced',
        n_estimators=100,
        max_depth=None,
        min_samples_leaf=1
)

final_model.fit(features_train_encoded, target_train)

In [ ]:
# Generate predictions for the test set
test_predictions = final_model.predict(features_test_encoded)

test_probabilities = final_model.predict_proba(
    features_test_encoded
)[:, 1]

In [ ]:
# Calculate and display test metrics
test_f1 = f1_score(target_test, test_predictions, zero_division=0)

test_results = pd.DataFrame({
    'accuracy': [accuracy_score(target_test, test_predictions)],
    'precision': [precision_score(target_test, test_predictions, zero_division=0)],
    'recall': [recall_score(target_test, test_predictions, zero_division=0)],
    'f1': [test_f1],
    'roc_auc': [roc_auc_score(target_test, test_probabilities)]
}, index=['Final random forest'])

display(test_results.round(3))

In [ ]:
# Build the confusion matrix
test_matrix = pd.DataFrame(
    confusion_matrix(
        target_test,
        test_predictions,
        labels=[0, 1]
    ),
    index=['Actually Stayed', 'Actually Left'],
    columns=['Predicted Stayed', 'Predicted Left']
)

display(test_matrix)

## Overall Conclusion

The objective of this project was to develop a binary classification model to identify bank customers at higher risk of leaving, achieving an F1-score above **0.59**.

**Data preparation and exploration**

The dataset contains 10,000 customer records, of which approximately **20.37%** belong to the churn class. This imbalance makes it important to evaluate models using metrics beyond accuracy.

The workflow included data cleaning, column-name standardization, missing-value treatment, one-hot encoding, and numerical feature scaling where appropriate. The dataset was split using stratification into training (60%), validation (20%), and test (20%) subsets, preserving the target distribution. Undersampling was implemented using pandas to reduce the majority class to approximately the size of the minority class.

**Model comparison and optimization**

Two algorithms were evaluated: Logistic Regression and Random Forest. Without class-imbalance treatment, Random Forest performed better, achieving a validation F1-score of **0.581**, compared with **0.289** for Logistic Regression.

Three imbalance-handling strategies were compared: oversampling, undersampling, and class weighting. A hyperparameter search was then conducted for both models. Logistic Regression improved but reached a best validation F1-score of only **0.497** (balanced class weights, `C=0.1`). Random Forest exceeded the required threshold in several configurations. The selected model used balanced class weights, `n_estimators=100`, `max_depth=None`, and `min_samples_leaf=1`, achieving validation F1-score **0.634**, precision **0.641**, recall **0.627**, and ROC-AUC **0.847**.

**Final test results**

The final model was a Random Forest classifier with `class_weight='balanced'`, `n_estimators=100`, `max_depth=None`, and `min_samples_leaf=1`. On the held-out test set, it achieved accuracy **0.849**, precision **0.635**, recall **0.613**, F1-score **0.623**, and ROC-AUC **0.869**.

The confusion matrix shows that the model correctly identified **250 of the 408 customers who left**. It missed **158** churners and incorrectly flagged **144** customers who stayed.

The test F1-score (**0.623**) was slightly lower than the validation F1-score (**0.634**). The configuration was selected based on validation performance among multiple candidates, so a modest difference on unseen test data is unsurprising.

**Business implications**

The project achieved its objective: test F1-score **0.623** exceeds the minimum requirement of **0.59**. The model identifies approximately **61.3%** of customers who leave, while producing both false positives and false negatives.

This analysis demonstrates why class imbalance and multiple evaluation metrics matter in churn prediction. The model could support early identification of at-risk customers and help prioritize retention activities.

Before operational use, the bank should assess the business costs of false positives and false negatives, optimize the classification threshold according to retention priorities, and validate performance on newer, representative data.

**Reproducibility**

The original notebook reports Python **3.12.10**. Package versions and results may vary between environments; verify the installed scikit-learn version before citing it as an exact reproduction requirement.